# Phase 11 — Combien de temps ça a duré

## Objectifs

- Construire une durée exploitable en secondes pour le plus grand nombre de relevés.
- Exploiter `duration_hours_min` lorsque `duration_seconds` est absente ou incohérente.
- Ne supprimer aucune ligne.
- Compter les durées restant inutilisables, les contradictions et les durées de plus d'une journée.
- Examiner les trois durées les plus longues et prendre une décision explicite.


## 1. Imports

In [ ]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd


## 2. Chemins et colonnes

In [ ]:
DATA_PATH = Path("../data/releves_klaxo3.csv")
OUTPUT_DIR = Path("../outputs")
PHASE11_DIR = OUTPUT_DIR / "phase_11_durees"
PHASE11_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]


## 3. Chargement robuste

In [ ]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row,
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)
print(f"Lignes chargées : {len(df)}")


## 4. Préparation des deux colonnes de durée

La colonne `duration_seconds` est convertie en nombre. La colonne `duration_hours_min` est conservée sous forme de texte normalisé afin d'en extraire une durée lorsque cela est possible.

In [ ]:
df["duration_seconds_original"] = df["duration_seconds"]
df["duration_seconds"] = pd.to_numeric(
    df["duration_seconds"],
    errors="coerce",
)

df["duration_hours_min_clean"] = (
    df["duration_hours_min"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


## 5. Fonction de conversion du texte vers des secondes

Cette fonction couvre les formes les plus fréquentes : secondes, minutes, heures, jours, plages de valeurs et expressions simples comme `half hour`. Les formats non interprétables retournent `NaN`.


In [ ]:
def parse_duration_to_seconds(value):
    texte = str(value).lower().strip()

    if texte in {"", "nan", "none", "unknown", "n/a", "na"}:
        return np.nan

    texte = texte.replace(",", ".")
    texte = re.sub(r"\babout\b|\bapprox\b|\bapproximately\b|\baround\b|\bover\b|\bunder\b", "", texte)
    texte = texte.replace("half an hour", "0.5 hour")
    texte = texte.replace("half hour", "0.5 hour")
    texte = texte.replace("a few minutes", "3 minutes")

    nombres = re.findall(r"\d+(?:\.\d+)?", texte)
    if not nombres:
        return np.nan

    valeurs = [float(nombre) for nombre in nombres]
    valeur = sum(valeurs) / len(valeurs)

    if re.search(r"day|days", texte):
        return valeur * 86400
    if re.search(r"hour|hours|hr|hrs", texte):
        return valeur * 3600
    if re.search(r"minute|minutes|min|mins", texte):
        return valeur * 60
    if re.search(r"second|seconds|sec|secs", texte):
        return valeur

    return np.nan


## 6. Extraction d'une durée depuis le texte

In [ ]:
df["duration_from_text_seconds"] = df["duration_hours_min_clean"].apply(
    parse_duration_to_seconds
)

df[[
    "duration_hours_min",
    "duration_seconds",
    "duration_from_text_seconds",
]].head(20)


## 7. Définition des contradictions

Une contradiction est enregistrée lorsque les deux colonnes fournissent une durée positive mais qu'elles diffèrent fortement. Le seuil retenu est un rapport supérieur ou égal à 2 entre les deux durées. Les cas où `duration_seconds` vaut 0 alors que le texte fournit une durée positive sont également considérés comme contradictoires.

In [ ]:
duree_numerique_positive = df["duration_seconds"].notna() & (df["duration_seconds"] > 0)
duree_texte_positive = df["duration_from_text_seconds"].notna() & (df["duration_from_text_seconds"] > 0)

rapport_durees = (
    df["duration_seconds"]
    / df["duration_from_text_seconds"]
)

contradiction_forte = (
    duree_numerique_positive
    & duree_texte_positive
    & ((rapport_durees >= 2) | (rapport_durees <= 0.5))
)

zero_contre_texte = (
    (df["duration_seconds"] == 0)
    & duree_texte_positive
)

df["durees_contradictoires"] = (
    contradiction_forte | zero_contre_texte
)

print(f"Contradictions détectées : {int(df['durees_contradictoires'].sum())}")


## 8. Construction de la durée finale

Règle retenue :

- une durée numérique strictement positive est utilisée lorsqu'elle n'est pas contradictoire ;
- une durée extraite du texte est utilisée si la durée numérique est absente, nulle, négative ou contradictoire ;
- sinon, la durée finale reste manquante.

Aucune ligne n'est supprimée.

In [ ]:
duree_numerique_valide = df["duration_seconds"].notna() & (df["duration_seconds"] > 0)

df["duration_finale_seconds"] = np.where(
    duree_numerique_valide & ~df["durees_contradictoires"],
    df["duration_seconds"],
    df["duration_from_text_seconds"],
)

df["source_duree_finale"] = np.select(
    [
        duree_numerique_valide & ~df["durees_contradictoires"],
        df["duration_from_text_seconds"].notna(),
    ],
    [
        "duration_seconds",
        "duration_hours_min_parsee",
    ],
    default="inutilisable",
)

assert len(df) == len(lignes_valides)


## 9. Statistiques demandées

In [ ]:
nombre_durees_inutilisables = int(df["duration_finale_seconds"].isna().sum())
nombre_contradictions = int(df["durees_contradictoires"].sum())
duree_mediane = df["duration_finale_seconds"].median()
nombre_plus_une_journee = int((df["duration_finale_seconds"] > 86400).sum())

resume_durees = pd.DataFrame([
    {
        "nombre_lignes_avant": len(lignes_valides),
        "nombre_lignes_apres": len(df),
        "durees_inutilisables": nombre_durees_inutilisables,
        "durees_contradictoires": nombre_contradictions,
        "duree_mediane_secondes": duree_mediane,
        "duree_mediane_minutes": duree_mediane / 60 if pd.notna(duree_mediane) else np.nan,
        "durees_plus_une_journee": nombre_plus_une_journee,
    }
])

resume_durees


## 10. Analyse des natures d'aberrations

Deux types d'aberrations sont comptés : les contradictions entre les deux colonnes et les durées supérieures à une journée.

In [ ]:
aberrations_durees = pd.DataFrame([
    {
        "type_aberration": "Deux colonnes de durée contradictoires",
        "nombre": nombre_contradictions,
    },
    {
        "type_aberration": "Durée finale supérieure à une journée",
        "nombre": nombre_plus_une_journee,
    },
    {
        "type_aberration": "Durée finale inutilisable",
        "nombre": nombre_durees_inutilisables,
    },
])

aberrations_durees


## 11. Exemples de contradictions

In [ ]:
exemples_contradictions = df.loc[
    df["durees_contradictoires"],
    [
        "datetime", "city", "duration_seconds",
        "duration_hours_min", "duration_from_text_seconds",
        "duration_finale_seconds", "source_duree_finale",
    ],
]

exemples_contradictions.head(20)


## 12. Trois durées finales les plus longues

Les durées extrêmement élevées sont affichées mais ne sont pas supprimées. Elles sont conservées dans le fichier et dans la colonne finale, car la décision retenue est de ne pas imposer de plafond arbitraire sans connaissance métier démontrant qu'une observation longue est impossible.

In [ ]:
trois_durees_plus_longues = df.loc[
    df["duration_finale_seconds"].notna(),
    [
        "datetime", "city", "country", "shape",
        "duration_seconds", "duration_hours_min",
        "duration_from_text_seconds", "duration_finale_seconds",
        "source_duree_finale", "comments",
    ],
]

trois_durees_plus_longues = trois_durees_plus_longues.nlargest(
    3,
    "duration_finale_seconds",
)

trois_durees_plus_longues


## 13. Export des résultats

In [ ]:
resume_durees.to_csv(
    PHASE11_DIR / "resume_durees.csv",
    index=False,
)

aberrations_durees.to_csv(
    PHASE11_DIR / "aberrations_durees.csv",
    index=False,
)

exemples_contradictions.to_csv(
    PHASE11_DIR / "exemples_durees_contradictoires.csv",
    index=False,
)

trois_durees_plus_longues.to_csv(
    PHASE11_DIR / "trois_durees_plus_longues.csv",
    index=False,
)

df[[
    "duration_seconds", "duration_hours_min",
    "duration_from_text_seconds", "duration_finale_seconds",
    "source_duree_finale", "durees_contradictoires",
]].to_csv(
    PHASE11_DIR / "durees_traitees.csv",
    index=False,
)

print(PHASE11_DIR)
